# Kaggle runner v3 — Feature selection only

Gắn một Kaggle Input chứa trực tiếp `experiment-data-perm/`, `experiment-data-geo-target/` và `split_manifests/`. Chỉ chạy Phase A; dùng notebook classification v3 riêng để đánh giá các panel đã lưu.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

# Sửa slug này theo Kaggle Input của bạn. Thư mục phải có ba nhánh đúng như ảnh.
INPUT_V3_DATASET = Path('/kaggle/input/experiment-data-v3')
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
WORK_ROOT = Path('/kaggle/working/kltn_data')
COHORTS = ['tcga_luad_lusc_primary_tumor', 'gse10245', 'gse30219', 'gse37745', 'gse42127']
K_GRID = [1, 2, 3, 4, 5, 10, 25, 50]

# Bật một, hai hoặc cả ba nhóm. Cả ba dùng cùng cohort, split 5x5 và K_GRID.
RUN_FILTER = True
RUN_EMBEDDED = True
RUN_WRAPPERS = False
FILTER_METHODS = ['f_test', 'mutual_info', 'mrmr', 'relieff']
EMBEDDED_METHODS = ['l1_lr', 'elastic_net', 'rf_importance', 'svm_rfe']
WRAPPER_METHODS = ['ga', 'bpso']
REPEATS = [0, 1, 2, 3, 4]
PREVIOUS_PROGRESS_ARCHIVE = None  # Kaggle Dataset ZIP của output Phase A trước đó, nếu resume
assert RUN_FILTER or RUN_EMBEDDED or RUN_WRAPPERS, 'Bật ít nhất một nhóm phương pháp'


In [ ]:
# Gate 1 — kiểm tra Input v3, split chính thức và 25 outer folds/cohort.
assert INPUT_V3_DATASET.is_dir(), f'Không thấy Kaggle Input: {INPUT_V3_DATASET}'
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    assert (INPUT_V3_DATASET / folder).is_dir(), f'Thiếu {folder}'
for cohort in COHORTS:
    branch = 'experiment-data-perm' if cohort.startswith('tcga_') else 'experiment-data-geo-target'
    cohort_dir = INPUT_V3_DATASET / branch / cohort
    for filename in ('expression.h5', 'metadata.csv', 'repeated_splits.csv'):
        assert (cohort_dir / filename).is_file(), f'Thiếu {cohort}/{filename}'
print('INPUT_V3_OK')


In [ ]:
# Gate 2 — clone code đã push; symlink Input read-only vào work root để results/ ghi được.
REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO_DIR, check=True)
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    link = WORK_ROOT / folder
    if not link.exists(): link.symlink_to(INPUT_V3_DATASET / folder, target_is_directory=True)
    assert link.resolve() == (INPUT_V3_DATASET / folder).resolve()
if PREVIOUS_PROGRESS_ARCHIVE is not None:
    shutil.unpack_archive(str(PREVIOUS_PROGRESS_ARCHIVE), str(WORK_ROOT))
os.environ['KLTN_DATA_ROOT'] = str(WORK_ROOT)
print('COMMIT', commit_hash, 'DATA_ROOT', WORK_ROOT)


In [ ]:
# Phase A only. Các nhóm đã bật ghi checkpoint/panel cùng E1_feature_selection để Phase B đọc chung.
methods = []
if RUN_FILTER: methods += FILTER_METHODS
if RUN_EMBEDDED: methods += EMBEDDED_METHODS
if RUN_WRAPPERS: methods += WRAPPER_METHODS
command = [sys.executable, 'scripts/run_e1_feature_selection.py', '--datasets', *COHORTS, '--methods', *methods, '--repeats', *map(str, REPEATS), '--k', *map(str, K_GRID)]
subprocess.run(command, cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# Archive chỉ Phase A; Save Version hoặc tạo Kaggle Dataset từ output này trước khi chạy notebook classification.
result_root = WORK_ROOT / 'results' / 'survey_v2' / 'E1_feature_selection'
assert any(result_root.glob('*/selected_genes.csv')), 'Không có panel feature selection'
archive = Path('/kaggle/working/e1_feature_selection_v3_output.zip')
shutil.make_archive(str(archive.with_suffix('')), 'zip', root_dir=WORK_ROOT, base_dir='results')
print('PHASE_A_OK', result_root, 'ARCHIVE=', archive)
